# Phase 3 · S5 — Precompute `demo_recs.json` cho web demo (Tier 2 live-rec)

Sinh dữ liệu cho mục **Live Recommendation Explorer**: với N user mẫu, lấy **top-K** gợi ý của
**chainRec / ALS / hybrid** (full catalog, mask **train**-seen để giữ test GT có thể trúng), kèm
**lịch sử đọc** và **ground-truth** (sách user thực sự đánh giá cao ở test). Tên sách stream từ UCSD
`goodreads_books.json.gz` (chỉ giữ id cần).

Output `demo_recs.json` → đặt cạnh `goodreads/results_demo.html`. **Cần Internet On** (tải sách metadata).


## 0 · Setup

In [ ]:
import os, json, time, pickle, gzip, urllib.request
from pathlib import Path
from dataclasses import dataclass
from typing import Literal
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
try:
    import implicit
except ImportError:
    os.system("pip install -q implicit"); import implicit
from scipy.sparse import csr_matrix
SEED=42; np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE="cuda" if torch.cuda.is_available() else "cpu"; print("Device:",DEVICE)
HF_TOKEN=None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN=UserSecretsClient().get_secret("HF_TOKEN")
except Exception: pass

N_DEMO_USERS=200; TOPK=8; HIST=6; ALPHA_HYB=0.5

## 1 · Load artifacts

In [ ]:
HF_REPO="vngclinh/goodreads-preprocessed"
PROC=Path("/kaggle/working/processed"); CK=Path("/kaggle/working/chainrec")
S5=Path("/kaggle/working/s5"); S5.mkdir(parents=True,exist_ok=True)
from huggingface_hub import hf_hub_download, HfApi
def _hf(r): return hf_hub_download(HF_REPO,r,repo_type="dataset",token=HF_TOKEN)
def lnpy(n):
    p=PROC/n; return np.load(p if p.exists() else _hf(f"chainrec/processed/{n}"))
def lpkl(n):
    p=PROC/n; return pickle.load(open(p if p.exists() else _hf(f"chainrec/processed/{n}"),"rb"))
def lmeta():
    p=PROC/"meta.json"; return json.loads(Path(p if p.exists() else _hf("chainrec/processed/meta.json")).read_text())
def ckpt(s,tag=""):
    fn=f"chainrec_{s}{tag}.pt"; p=CK/fn
    return str(p) if p.exists() else _hf(f"chainrec/{fn}")

data_train=lnpy("data_train.npy"); data_test=lnpy("data_test.npy")
meta=lmeta(); N_ITEM,N_USER,N_STAGE=meta["n_item"],meta["n_user"],meta["n_stage"]; REC=N_STAGE-1
bridge=pickle.load(open(_hf("s2/id_bridge.pkl"),"rb"))
idx2book=np.asarray(bridge["idx2book_str"],dtype=object)
print(f"n_user={N_USER:,} n_item={N_ITEM:,} REC={REC}")

## 2 · Model + scorers

In [ ]:
@dataclass
class Cfg:
    n_user:int;n_item:int;n_stage:int=4;embed_dim:int=16;beta:float=1.0;learn_beta:bool=True
    l2:float=0.01;lr:float=0.001;sampler:str="stagewise";device:str=DEVICE
class ChainRec(nn.Module):
    def __init__(s,c):
        super().__init__();s.cfg=c;K,L=c.embed_dim,c.n_stage
        s.user_emb=nn.Embedding(c.n_user,K);s.item_emb=nn.Embedding(c.n_item,K);s.stage_emb=nn.Embedding(L,K)
        s.b0=nn.Parameter(torch.zeros(1));s.b_user=nn.Embedding(c.n_user,1);s.b_item=nn.Embedding(c.n_item,1)
        lb=torch.log(torch.tensor(float(c.beta)))
        s.log_beta=nn.Parameter(lb) if c.learn_beta else s.register_buffer("log_beta",lb)
    @property
    def beta(s): return torch.clamp(s.log_beta.exp(),min=1.0)
    def _rect(s,d): b=s.beta; return F.softplus(b*d)/b
m_cr=ChainRec(Cfg(n_user=N_USER,n_item=N_ITEM,n_stage=N_STAGE)).to(DEVICE)
try: m_cr.load_state_dict(torch.load(ckpt("stagewise","_r10"),map_location=DEVICE))
except Exception: m_cr.load_state_dict(torch.load(ckpt("stagewise"),map_location=DEVICE))
m_cr.eval()

@torch.no_grad()
def cr_score(u):
    ie=m_cr.item_emb.weight; bi=m_cr.b_item.weight.squeeze(-1); sw=m_cr.stage_emb.weight
    uv=m_cr.user_emb(u); bias=(m_cr.b0+m_cr.b_user(u).squeeze(-1)).unsqueeze(1)
    acc=torch.zeros(u.shape[0],ie.shape[0],device=u.device)
    for l in range(REC,m_cr.cfg.n_stage): acc=acc+m_cr._rect((uv*sw[l].unsqueeze(0))@ie.t())
    return bias+bi.unsqueeze(0)+acc

## 3 · Train ALS (~60s) + scorers ALS/hybrid

In [ ]:
pos=data_train[data_train[:,2]==REC]
M=csr_matrix((np.ones(len(pos),np.float32),(pos[:,0].astype(np.int32),pos[:,1].astype(np.int32))),shape=(N_USER,N_ITEM))
try: als=implicit.als.AlternatingLeastSquares(factors=64,iterations=20,regularization=0.1,alpha=40,random_state=SEED,use_gpu=False)
except TypeError: als=implicit.als.AlternatingLeastSquares(factors=64,iterations=20,regularization=0.1,random_state=SEED,use_gpu=False); M=M*40.0
t0=time.time(); als.fit(M); print("ALS",round(time.time()-t0,1),"s")
U=torch.tensor(np.asarray(als.user_factors,np.float32),device=DEVICE)
V=torch.tensor(np.asarray(als.item_factors,np.float32),device=DEVICE)
@torch.no_grad()
def als_score(u): return U[u]@V.t()
@torch.no_grad()
def hyb_score(u):
    def z(x): return (x-x.mean(1,keepdim=True))/(x.std(1,keepdim=True)+1e-8)
    return ALPHA_HYB*z(als_score(u))+(1-ALPHA_HYB)*z(cr_score(u))
SCORERS={"chainRec":cr_score,"ALS":als_score,"hybrid":hyb_score}

## 4 · Chọn user mẫu + top-K mỗi model (mask train-seen, giữ test GT trúng được)

In [ ]:
rng=np.random.default_rng(SEED)
# train-seen (mask) + history theo user, chỉ cho user có test recommend
test_rec=data_test[data_test[:,2]==REC]
gt_by_user={}
for u,i,_ in test_rec: gt_by_user.setdefault(int(u),set()).add(int(i))
cand=np.array(sorted(gt_by_user)); rng.shuffle(cand)

# train items theo user (chỉ user ứng viên) để mask + lịch sử
cand_set=set(cand.tolist())
train_by_user={}
for u,i,_ in data_train:
    u=int(u)
    if u in cand_set: train_by_user.setdefault(u,[]).append(int(i))

users=[u for u in cand if len(train_by_user.get(u,[]))>=5][:N_DEMO_USERS]
print("demo users:",len(users))

@torch.no_grad()
def topk(score_fn,uids,k,batch=32):
    res={}
    for st in range(0,len(uids),batch):
        ch=uids[st:st+batch]; u=torch.tensor(ch,dtype=torch.long,device=DEVICE); sc=score_fn(u)
        for b,uu in enumerate(ch):
            seen=train_by_user.get(int(uu),[])
            if seen: sc[b,torch.tensor(seen,dtype=torch.long,device=DEVICE)]=float("-inf")
            res[int(uu)]=torch.topk(sc[b],k).indices.cpu().numpy().tolist()
        del sc
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    return res

recs={name:topk(fn,users,TOPK) for name,fn in SCORERS.items()}
print("scored.")
# gom item idx cần tên
need_idx=set()
for u in users:
    need_idx.update(train_by_user[u][:HIST]); need_idx.update(gt_by_user[u])
    for name in SCORERS: need_idx.update(recs[name][u])
need_book=set(str(idx2book[i]) for i in need_idx if idx2book[i] is not None)
print("cần tên cho",len(need_book),"sách")

## 5 · Stream tên sách từ UCSD `goodreads_books.json.gz` (chỉ giữ id cần)

In [ ]:
BOOKS_URL="https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/goodreads_books.json.gz"
dst=Path("/kaggle/working/goodreads_books.json.gz")
if not dst.exists():
    print("downloading goodreads_books.json.gz (~2GB) ..."); urllib.request.urlretrieve(BOOKS_URL,dst)
title={}; t0=time.time()
with gzip.open(dst,"rt",encoding="utf-8") as f:
    for ln in f:
        try: d=json.loads(ln)
        except Exception: continue
        b=d.get("book_id")
        if b in need_book:
            title[b]={"title":(d.get("title") or d.get("title_without_series") or f"Book {b}").strip(),
                      "img":d.get("image_url","")}
            if len(title)==len(need_book): break
print(f"resolved {len(title)}/{len(need_book)} titles in {time.time()-t0:.0f}s")
def info(idx):
    b=idx2book[idx]
    if b is None: return {"title":f"item#{idx}","img":"","book_id":None}
    b=str(b); t=title.get(b,{"title":f"Book {b}","img":""})
    return {"title":t["title"],"img":t.get("img",""),"book_id":b}

## 6 · Lắp `demo_recs.json` + push HF

In [ ]:
out={"meta":{"n_users":len(users),"topk":TOPK,"alpha_hybrid":ALPHA_HYB,
      "note":"Demo gợi ý full-catalog (1.57M), mask train-seen; minh hoạ hành vi model — không phải bảng full-ranking."},
      "users":[]}
for u in users:
    gt=gt_by_user[u]
    rec={}
    for name in SCORERS:
        rec[name]=[{**info(i),"hit":(i in gt)} for i in recs[name][u]]
    out["users"].append({
        "uid":int(u),
        "history":[info(i) for i in train_by_user[u][:HIST]],
        "truth":[info(i) for i in list(gt)[:5]],
        "models":rec})
p=S5/"demo_recs.json"; json.dump(out,open(p,"w"),ensure_ascii=False)
print("wrote",p, round(p.stat().st_size/1e3,1),"KB")
if HF_TOKEN:
    HfApi().upload_file(path_or_fileobj=str(p),path_in_repo="s5/demo_recs.json",
                        repo_id=HF_REPO,repo_type="dataset",token=HF_TOKEN); print("pushed s5/demo_recs.json")

## 7 · Gắn vào web demo

Tải `demo_recs.json` về (từ `/kaggle/working/s5/` hoặc HF `s5/demo_recs.json`) và **đặt cạnh**
`goodreads/results_demo.html`. Mục *Live Recommendation Explorer* sẽ tự `fetch('demo_recs.json')`;
nếu mở bằng double-click (file://) bị chặn CORS thì dùng nút **"Tải demo_recs.json"** để chọn file thủ công.
